In [ ]:
--@exclude_input=xyf_fengkong.api_activation_base
--@exclude_input=xyf_dwd.dwd_inloan_leap_vip_order_hf
--@exclude_input=xyf_dwd.dwd_event_tracking_log_di
--@exclude_input=xyf_dwd.dwd_user_vip_order_df
--@exclude_input=xyf_dwd.dwd_inloan_t_decision_result_detail_df
--@exclude_input=xyf_fengkong.sd_app_activation_amtlevel
--@exclude_input=xyf_dwd.dwd_sfy_sta_channel_df_v
--@exclude_input=xyf_dwd.dwd_xyf_bi_cash_activation_log_df_v
--@exclude_input=xyf_bi_dev.utm_source_channel_v1_cdf_v
--@exclude_input=xyf_dim.dim_backup_ods_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_3_0_v2_df
--@exclude_input=xyf_dim.dim_backup_ods_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_1_0_v2_df
--@exclude_input=xyf_dim.dim_backup_ods_xinyongfei_model_personalloan_sd_line_amtlevel_rh_app_v1_v2_df
--@exclude_input=xyf_fengkong.personalloan_sd_line_amtlevel_rh_app_v4_a_score_apply
--@exclude_input=xyf_ods.ods_sfy_sta_channel_df
--@exclude_input=xyf_dwd.dwd_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_3_0_v2_df_v
--@exclude_input=xyf_dim.dim_pub_date
--@exclude_input=xyf_ods.ods_xinyongfei_model_personalloan_sd_line_amtlevel_rh_app_v1_v2_df
--@exclude_input=xyf_ods.ods_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_1_0_v2_df
--odps sql 
--********************************************************************--
--author:王雪勤
--create time:2023-11-08 11:11:35
--********************************************************************--
DROP TABLE IF EXISTS xyf_bi_dev.bi_app_zhuanhua_shouxin_xujia_uid_T0
;

CREATE TABLE xyf_bi_dev.bi_app_zhuanhua_shouxin_xujia_uid_T0 AS
WITH wanjian_userno AS 
(
    SELECT  shenqing.* ---,ROW_NUMBER() OVER (PARTITION BY cust_no ORDER BY created_time ASC ) AS is_首申
    FROM    (
                SELECT  *
                FROM    xyf_dwd.dwd_preloan_credit_apply_df
                WHERE   pt = '${bizdate}'
                AND     date(created_time) >= '2024-01-01'
                AND     app IN ('xyf01')
                AND     inner_app IN ('xyf01','xyf01_hrui02','xyf01_xcjr','xyf01_hrui01','xyf01_alyxy','xyf01_alygd','xyf01_alyfz','xyf01_zyxj01','xyf01_zyxjwld01','xyf01_zyxjzl01','xyf01_elm')
                AND     NVL(app_activation_type,'') <> 'loan_recredit_activation'
                QUALIFY ROW_NUMBER() OVER (PARTITION BY user_no,date(created_time) ORDER BY created_time DESC ) = 1
            ) shenqing
    LEFT ANTI JOIN  (
                        SELECT  DISTINCT biz_flow_number
                        FROM    xyf_dwd.dwd_inloan_t_decision_result_detail_df
                        WHERE   enginecode = 'jcl_20240722000003'
                        AND     pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
                        AND     inner_app <> 'xyf01_test1'
                    ) b
    ON      shenqing.biz_flow_number = b.biz_flow_number
)


SELECT  TO_DATE(wanjian.created_time) dt
        ,wanjian.created_time
        ,zhou.week_range AS wt
        ,SUBSTR(wanjian.created_time,1,7) mt
        ,CASE   WHEN 0 <= MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) < 1000 THEN 'A. [0k-1k)'
                WHEN 1000 <= MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) < 3000 THEN 'B.[1k,3k)'
                WHEN 3000 <= MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 5000 THEN 'C.[3k,5k]'
                WHEN 5000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 10000 THEN 'D. (5k,1w]'
                WHEN 10000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 20000 THEN 'E. (1w,2w]'
                WHEN 20000 < MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) < 50000 THEN 'F. (2w,5w)'
                WHEN 50000 <= MAX(shouxin.init_credit_line / 100) AND MAX(shouxin.init_credit_line / 100) <= 100000 THEN 'G. [5w,10w]'
                WHEN 100000 < MAX(shouxin.init_credit_line / 100) THEN 'H.10w+'
                ELSE ''
        END AS 授信金额_level
        ,qu_zhu.first_channel AS biz_type_bi
        ,qu_gui.first_channel AS biz_type_bi_3
        ,MAX(CASE    WHEN xujia_type.人行评级 < 0 THEN api_rk.人行评级 ELSE xujia_type.人行评级 END) AS 实际执行人行完件评级_v3_v4_xifen
        ,CASE   WHEN DATEDIFF(date(wanjian.created_time),MIN(c.register_time)) = 0 THEN '0d'
                WHEN DATEDIFF(date(wanjian.created_time),MIN(c.register_time)) BETWEEN 1 AND 30 THEN '1-30d'
                WHEN DATEDIFF(date(wanjian.created_time),MIN(c.register_time)) BETWEEN 31 AND 60 THEN '31d-60d'
                WHEN DATEDIFF(date(wanjian.created_time),MIN(c.register_time)) > 60 THEN '61d+'
                ELSE ''
        END AS 注册授信时间差
        ,wanjian.user_no
        ,c.reg_channel AS 注册渠道_产品口径
        ,is_API半流程
        ,is_虚假给额
        ,CASE   WHEN wanjian.本次授信前最早一次失败时间 IS NOT NULL THEN '非首申_风险口径'
                ELSE '首申_风险口径'
        END AS 是否首申_风险
        ,CASE   WHEN wanjian.本次授信前最早一次完件时间 IS NOT NULL THEN '非首申_业务口径'
                ELSE '首申_业务口径'
        END AS 是否首申_业务
        ,CASE   WHEN rand_num <= 9 THEN '大盘对照组'
                WHEN rand_num <= 19 THEN '电销对照组'
                ELSE '电销营销组'
        END AS 大盘随机数
        ,phone_brand
        ,MAX(c.first_login_version) 登录版本
        ,MAX(shouxin.init_credit_line / 100) AS 授信额度_shu
        ,COUNT(DISTINCT CASE    WHEN po.cust_no IS NOT NULL THEN shouxin.user_no END) 是否通过预借款发起
        ,COUNT(DISTINCT CASE    WHEN xujia_type.是否api虚假给额 = 1 THEN shouxin.user_no END) API虚假给额
        ,count(distinct case when shouxin.is_虚假给额 = '虚假给额' and is_API半流程 = 'API半流程' then shouxin.user_no end)半流程虚假给额
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN shouxin.user_no END
        ) 提现人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 THEN shouxin.user_no END
        ) T7提现人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 THEN shouxin.user_no END
        ) T30提现人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 THEN shouxin.user_no END
        ) 累积提现人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND vip.order_time IS NOT NULL THEN shouxin.user_no END
        ) T0会员卡签约
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND vip.status = 3 THEN shouxin.user_no END
        ) T0会员卡扣款
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND vip.status = 3 THEN vip.real_card_price / 100 END
        ) T0会员卡扣款金额
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND vip.order_time IS NOT NULL THEN shouxin.user_no END
        ) T7会员卡签约
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND vip.status = 3 THEN shouxin.user_no END
        ) T7会员卡扣款
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND vip.status = 3 THEN vip.real_card_price / 100 END
        ) T7会员卡扣款金额
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND vip.order_time IS NOT NULL THEN shouxin.user_no END
        ) T30会员卡签约
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND vip.status = 3 THEN shouxin.user_no END
        ) T30会员卡扣款
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND vip.status = 3 THEN vip.real_card_price / 100 END
        ) T30会员卡扣款金额
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND vip.order_time IS NOT NULL THEN shouxin.user_no END
        ) 累积会员卡签约
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND vip.status = 3 THEN shouxin.user_no END
        ) 累积会员卡扣款
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND vip.status = 3 THEN vip.real_card_price / 100 END
        ) 累积会员卡扣款金额
        ,MAX(CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN o.order_amt END) 提现金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 THEN o.order_amt END
        ) T7提现金额 ---T7提现金额,T30提现金额,累积提现金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 THEN o.order_amt END
        ) T30提现金额
        ,MAX(CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 THEN o.order_amt END) 累积提现金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN o.order_amt * o.period END
        ) 提现金额_period
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.loan_amt END
        ) 放款金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.fee_rate * o.loan_amt END
        ) T0放款金额_定价
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN o.loan_amt END
        ) T7放款金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN o.loan_amt END
        ) T30放款金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN o.loan_amt END
        ) 累积放款金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN o.fee_rate * o.loan_amt END
        ) 累积放款金额_定价
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN o.loan_amt * o.period END
        ) 放款金额_period
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.risk_status = 'pass' THEN shouxin.user_no END
        ) 风控通过人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.risk_status = 'pass' THEN shouxin.user_no END
        ) T7风控通过人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.risk_status = 'pass' THEN shouxin.user_no END
        ) T30风控通过人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.risk_status = 'pass' THEN shouxin.user_no END
        ) 累积风控通过人
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.risk_status = 'pass' THEN o.order_amt END
        ) T0风险通过_提现金额 ----T7风险通过_提现金额
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.risk_status = 'pass' THEN o.order_amt END
        ) T7风险通过_提现金额
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN shouxin.user_no END
        ) 放款人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN shouxin.user_no END
        ) T7放款人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN shouxin.user_no END
        ) T30放款人
        ,COUNT(DISTINCT 
              CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN shouxin.user_no END
        ) 累积放款人
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) = 0 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END
        ) 授信额度_放款
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 7 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END
        ) 授信额度_放款7
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) BETWEEN 0 AND 30 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END
        ) 授信额度_放款30
        ,MAX(
            CASE    WHEN DATEDIFF(o.first_order_time,shouxin.created_time) >= 0 AND o.loan_time IS NOT NULL THEN shouxin.init_credit_line / 100 END
        ) 授信额度_放款累积
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账7plus金额 ELSE NULL END) 出账7plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账15plus金额 ELSE NULL END) 出账15plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账30plus金额 ELSE NULL END) 出账30plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账60plus金额 ELSE NULL END) 出账60plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账90plus金额 ELSE NULL END) 出账90plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账120plus金额 ELSE NULL END) 出账120plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账150plus金额 ELSE NULL END) 出账150plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账180plus金额 ELSE NULL END) 出账180plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账210plus金额 ELSE NULL END) 出账210plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账240plus金额 ELSE NULL END) 出账240plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账270plus金额 ELSE NULL END) 出账270plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账300plus金额 ELSE NULL END) 出账300plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账330plus金额 ELSE NULL END) 出账330plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账360plus金额 ELSE NULL END) 出账360plus金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账7plus逾期金额 ELSE NULL END) 出账7plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账15plus逾期金额 ELSE NULL END) 出账15plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账30plus逾期金额 ELSE NULL END) 出账30plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账60plus逾期金额 ELSE NULL END) 出账60plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账90plus逾期金额 ELSE NULL END) 出账90plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账120plus逾期金额 ELSE NULL END) 出账120plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账150plus逾期金额 ELSE NULL END) 出账150plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账180plus逾期金额 ELSE NULL END) 出账180plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账210plus逾期金额 ELSE NULL END) 出账210plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账240plus逾期金额 ELSE NULL END) 出账240plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账270plus逾期金额 ELSE NULL END) 出账270plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账300plus逾期金额 ELSE NULL END) 出账300plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账330plus逾期金额 ELSE NULL END) 出账330plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN 出账360plus逾期金额 ELSE NULL END) 出账360plus逾期金额
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN fpd7_available ELSE NULL END) fpd7_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN fpd15_available ELSE NULL END) fpd15_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m1_available ELSE NULL END) m1_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m2_available ELSE NULL END) m2_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m3_available ELSE NULL END) m3_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m4_available ELSE NULL END) m4_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m5_available ELSE NULL END) m5_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m6_available ELSE NULL END) m6_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m7_available ELSE NULL END) m7_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m8_available ELSE NULL END) m8_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m9_available ELSE NULL END) m9_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m10_available ELSE NULL END) m10_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m11_available ELSE NULL END) m11_available
        ,MAX(CASE    WHEN o.loan_time IS NOT NULL THEN m12_available ELSE NULL END) m12_available
        ,COUNT(DISTINCT CASE    WHEN back.user_no IS NOT NULL THEN shouxin.user_no END) back_cnt
        ,COUNT(DISTINCT CASE    WHEN core.md_zh = '提现页曝光' THEN shouxin.user_no END) 提现页_cnt
        ,COUNT(DISTINCT CASE    WHEN core.md_zh = '点击借款按钮' THEN shouxin.user_no END) 点击借款按钮_cnt
        ,COUNT(DISTINCT 
              CASE    WHEN core.md_zh = '点击借款按钮' AND DATEDIFF(o.first_order_time,shouxin.created_time) = 0 THEN shouxin.user_no ELSE NULL END
        ) AS create_cnt_dj
        ,COUNT(DISTINCT 
              CASE    WHEN core.md_zh = '点击借款按钮' AND t_jkfwxy.user_no IS NOT NULL THEN shouxin.user_no ELSE NULL END
        ) AS jkfwxy_cnt
        ,COUNT(DISTINCT 
              CASE    WHEN core.md_zh = '点击借款按钮' AND t_fxsq.user_no IS NOT NULL THEN shouxin.user_no ELSE NULL END
        ) AS fxsq_cnt
        ,COUNT(DISTINCT 
              CASE    WHEN core.md_zh = '点击借款按钮' AND t_vip.user_no IS NOT NULL THEN shouxin.user_no ELSE NULL END
        ) AS vip_cnt
        ,COUNT(DISTINCT 
              CASE    WHEN core.md_zh = '点击借款按钮' AND t_bj.user_no IS NOT NULL THEN shouxin.user_no ELSE NULL END
        ) AS bj_cnt
        ,COUNT(DISTINCT 
              CASE    WHEN core.md_zh = '点击借款按钮' AND t_yzm.user_no IS NOT NULL THEN shouxin.user_no ELSE NULL END
        ) AS yzm_cnt
        ,COUNT(DISTINCT 
              CASE    WHEN core.md_zh = '点击借款按钮' AND t_qrtc.user_no IS NOT NULL THEN shouxin.user_no ELSE NULL END
        ) AS qrtc_cnt
FROM    (
            SELECT  w.user_no
                    ,w.cust_no
                    ,w.created_time
                    ,CASE   WHEN w.inner_app IN ('xyf01') THEN CASE   WHEN w.client_code IN ('MPP001000068') THEN '微信小程序'
                                    WHEN w.client_code IN ('MPP002000069') THEN '抖音小程序'
                                    ELSE 'APP全流程'
                            END
                            ELSE 'API半流程'
                    END AS is_API半流程
                    ,CASE   WHEN date(w.created_time) <= '2024-01-05' THEN SUBSTR(conv2(sha2(CONCAT('xinkeyunying',w.user_no),256),16,10),-2)
                            WHEN date(w.created_time) <= '2025-01-22' THEN RANDOMV3('xinkeyunying',w.user_no,2)
                            ELSE RANDOMV3('xinkeyunying2025',w.user_no,2)
                    END AS rand_num
                    ,MAX(sxsb.created_time) 本次授信前最早一次失败时间
                    ,MAX(scsq.created_time) 本次授信前最早一次完件时间
            FROM    wanjian_userno w
            LEFT JOIN   (
                            SELECT  *
                            FROM    xyf_dwd.dwd_preloan_credit_apply_df
                            WHERE   pt = '${bizdate}' ---AND     date(created_time) >= '2023-01-01'
                            AND     app IN ('xyf01','fxk')
                            AND     NVL(app_activation_type,'') <> 'loan_recredit_activation'
                            AND     status = 3
                        ) sxsb
            ON      sxsb.cust_no = w.cust_no
            AND     sxsb.created_time < w.created_time
            LEFT JOIN   (
                            SELECT  *
                            FROM    xyf_dwd.dwd_preloan_credit_apply_df
                            WHERE   pt = '${bizdate}' ---AND     date(created_time) >= '2023-01-01'
                            AND     app IN ('xyf01','fxk')
                            AND     NVL(app_activation_type,'') <> 'loan_recredit_activation'
                        ) scsq
            ON      scsq.cust_no = w.cust_no
            AND     scsq.created_time < w.created_time
            GROUP BY w.user_no
                     ,w.cust_no
                     ,w.created_time
                     ,is_API半流程
                     ,rand_num
        ) wanjian
LEFT JOIN   (
                SELECT  shouxin.*
                        ,CASE   WHEN xujia.biz_flow_number IS NOT NULL AND inner_app IN ('xyf01','fxk') THEN '虚假给额'
                        when inner_app not in ('xyf01','fxk') and init_credit_line / 100 < 1000 then '虚假给额'  -----这里是半流程的虚假给额规则。写innerapp not in是因为!我这就只看APP和半流程的授信！
                                ELSE '非虚假给额'
                        END AS is_虚假给额
                FROM    (
                            SELECT  *
                            FROM    wanjian_userno
                            WHERE   status = 2 --成功
                        ) shouxin
                LEFT JOIN   (
                                -- 虚假给额的授信成功用户口径，biz_flow_number关联授信表
                                SELECT  biz_flow_number -- 授信biz_flow_number
                                FROM    xyf_dwd.dwd_inloan_t_decision_result_detail_df
                                WHERE   pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
                                AND     enginecode = 'jcl_20240923000003'
                                AND     GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") RLIKE 'fake_activation'
                                AND     decision_time >= '2024-10-15 00:00:00' --
                                -- 存在少量异常数据，同一个biz_flow_number+enginename 存在多条记录；下面排序做个兜底的清洗
                                QUALIFY ROW_NUMBER() OVER (PARTITION BY biz_flow_number,date(decision_time) ORDER BY decision_time DESC ) = 1
                            ) xujia
                ON      shouxin.biz_flow_number = xujia.biz_flow_number
            ) shouxin
ON      wanjian.user_no = shouxin.user_no
AND     DATEDIFF(wanjian.created_time,shouxin.created_time) = 0
LEFT JOIN   (
                SELECT  授信_biz_flow_number
                        ,是否api虚假给额
                        ,CAST(NVL(人行评级,-1.0) AS DOUBLE) 人行评级
                FROM    xyf_bi.aji_app_rank_activation_forbi
                QUALIFY ROW_NUMBER() OVER (PARTITION BY 授信_biz_flow_number ORDER BY created_time ) = 1 ---不知道需不需要去重，但这样保险一点
            ) xujia_type
ON      xujia_type.授信_biz_flow_number = shouxin.biz_flow_number
LEFT JOIN   (
                SELECT  biz_flow_number AS 授信_biz_flow_number
                        ,CAST(NVL(personalloan_sd_line_amtlevel_rh_api,-1.0) AS DOUBLE) 人行评级
                FROM    xyf_fengkong.api_activation_base
            ) api_rk
ON      api_rk.授信_biz_flow_number = shouxin.biz_flow_number
LEFT JOIN   (
                SELECT  day_id_iso
                        ,CONCAT(day_week01_xf_new,'至',day_weekend_xf_new) AS week_range
                FROM    xyf_dim.dim_pub_date
            ) zhou
ON      TO_DATE(wanjian.created_time) = zhou.day_id_iso
LEFT JOIN   (
                --归因渠道相关--产品侧注册渠道,
                SELECT  c.*
                        ,CASE   WHEN a.attribution_source IS NULL THEN c.current_utm_source
                                ELSE a.attribution_source
                        END AS attribution_source
                        ,CASE   WHEN c.register_utm_source IN ('QD-XXL-HW','QD-XXL-HW01','QD-XXL-IOS-SS','QD-XXL-VIVO') THEN '应用商店' --when lower(a.register_utm_source) like '%hw%' then 'Huawei'
                                WHEN c.register_utm_source LIKE '%QD-XXL-GDT-XCX%' THEN '小程序-信息流投放'
                                WHEN c.register_utm_source LIKE '%QD-CPP-DX-XYF01-XCX%' THEN '小程序-短信投放'
                                WHEN c.register_utm_source LIKE '%-XXL-%' THEN '信息流'
                                WHEN c.register_utm_source LIKE '%QD-CPP-DX%' THEN '短信'
                                WHEN c.register_utm_source IN ('信用飞APP','xyf_app') THEN '自然流量'
                                WHEN c.register_utm_source LIKE '%QD-CPP-XYF01-BD-APK%' THEN 'APK'
                                WHEN c.register_utm_source = 'xyf_mnp' THEN '小程序-自然流量'
                                WHEN c.register_utm_source LIKE '%QD-CPP-XYF01-BD-QLC%' THEN 'H5全流程'
                                WHEN c.register_utm_source LIKE '%QD-CPP-XYF01-%'
                                    OR c.register_utm_source LIKE '%QD-CPA-XYF01-%'
                                    OR c.register_utm_source LIKE '%QD-CPS-XYF01-%' THEN 'H5注册跳下载'
                                ELSE 'other'
                        END AS reg_channel
                        ,CASE   WHEN (lower(brand) LIKE '%huawei%'
                                    OR lower(brand) LIKE '%honor%'
                                    OR lower(brand) LIKE '%hinova%') THEN '华为'
                                WHEN brand LIKE '360%' THEN '360'
                                WHEN lower(brand) LIKE '%samsung%' THEN '三星'
                                WHEN (
                                            lower(brand) LIKE '%blackshark%'
                                                OR lower(brand) LIKE '%xiaomi%'
                                                OR lower(brand) LIKE '%redmi%'
                                                OR lower(brand) = 'mi'
                                ) THEN '小米'
                                WHEN lower(brand) LIKE '%lenovo%' THEN '联想'
                                WHEN lower(brand) LIKE '%meizu%' THEN '魅族'
                                WHEN lower(brand) LIKE '%nubia%' THEN '努比亚'
                                WHEN (
                                            lower(brand) LIKE '%oppo%'
                                                OR lower(brand) LIKE '%realme%'
                                                OR brand LIKE '%oneplus%'
                                ) THEN 'OPPO'
                                WHEN lower(brand) LIKE '%smartisan%' THEN '锤子'
                                WHEN lower(brand) LIKE '%vivo%' THEN 'VIVO'
                                WHEN brand LIKE '%8848%' THEN '8848'
                                WHEN lower(brand) LIKE '%iphone%' THEN 'iPhone'
                                WHEN brand NOT IN ('\N') OR brand IS NOT NULL THEN '安卓未获取品牌'
                                ELSE '未知'
                        END AS phone_brand
                FROM    (
                            SELECT  *
                                    ,first_mobile_brand AS brand
                                    ,current_utm_source AS register_utm_source
                            FROM    xyf_dws.dws_preloan_register_conversion_df
                            WHERE   pt = MAX_PT('xyf_dws.dws_preloan_register_conversion_df')
                            AND     app IN ('xyf','xyf01')
                        ) c
                LEFT JOIN   (
                                SELECT  user_id
                                        ,attribution_source
                                FROM    xyf_dwd.dwd_xyf_flow_sys_flow_attribution_result_dup_df
                                WHERE   pt = '${bizdate}'
                                QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_time DESC ) = 1
                            ) a
                ON      c.app_user_id = a.user_id
            ) c
ON      wanjian.user_no = c.app_user_id ---判断用户是否回到APP
LEFT JOIN   (
                SELECT  user_no
                        ,date(tracking_timestamp) dt
                        ,MAX(tracking_timestamp) event_time
                FROM    xyf_dwd.dwd_event_tracking_log_di ---WHERE   pt >= '20250901'
                WHERE   DATE(TO_DATE(pt,"yyyyMMdd")) >= DATE_SUB(CURRENT_TIMESTAMP(),180)
                GROUP BY user_no
                         ,dt
            ) back
ON      back.user_no = shouxin.user_no
AND     SUBSTR(back.dt,1,10) = SUBSTR(shouxin.created_time,1,10)
AND     shouxin.created_time < back.event_time
LEFT JOIN   (
                SELECT  user_no
                        ,date(tracking_timestamp) dt
                        ,md_zh
                        ,MIN(tracking_timestamp) event_time
                FROM    xyf_dwd.dwd_event_tracking_log_core_di
                WHERE   DATE(TO_DATE(pt,"yyyyMMdd")) >= DATE_SUB(CURRENT_TIMESTAMP(),180)
                AND     md_zh IN ('提现页曝光','点击借款按钮')
                GROUP BY user_no
                         ,dt
                         ,md_zh
            ) core
ON      core.user_no = shouxin.user_no
AND     SUBSTR(core.dt,1,10) = SUBSTR(shouxin.created_time,1,10)
LEFT JOIN   (
                SELECT  DISTINCT user_no
                        ,SUBSTR(tracking_timestamp,1,10) dt
                FROM    xyf_dwd.dwd_event_tracking_log_core_di
                WHERE   SUBSTR(TO_DATE(pt,"yyyyMMdd"),1,10) >= DATE_SUB(CURRENT_TIMESTAMP(),180)
                AND     tracking_id IN ('JK-52-56-0-286')
            ) t_jkfwxy ----借款及服务协议确认弹窗
ON      t_jkfwxy.user_no = shouxin.user_no
AND     t_jkfwxy.dt = SUBSTR(shouxin.created_time,1,10)
LEFT JOIN   (
                SELECT  DISTINCT user_no
                        ,SUBSTR(tracking_timestamp,1,10) dt
                FROM    xyf_dwd.dwd_event_tracking_log_core_di
                WHERE   SUBSTR(TO_DATE(pt,"yyyyMMdd"),1,10) >= DATE_SUB(CURRENT_TIMESTAMP(),180)
                AND     tracking_id IN ('JK-110-156-0-1168','XYF_H5_003055') ---缺'XYF_H5_003055'
            ) t_vip ----会员卡挽留弹窗
ON      t_vip.user_no = shouxin.user_no
AND     t_vip.dt = SUBSTR(shouxin.created_time,1,10)
LEFT JOIN   (
                SELECT  DISTINCT user_no
                        ,SUBSTR(tracking_timestamp,1,10) dt
                FROM    xyf_dwd.dwd_event_tracking_log_core_di
                WHERE   SUBSTR(TO_DATE(pt,"yyyyMMdd"),1,10) >= DATE_SUB(CURRENT_TIMESTAMP(),180)
                AND     tracking_id IN ('JK-52-58-0-302')
            ) t_fxsq ----风险授权
ON      t_fxsq.user_no = shouxin.user_no
AND     t_fxsq.dt = SUBSTR(shouxin.created_time,1,10)
LEFT JOIN   (
                SELECT  DISTINCT user_no
                        ,SUBSTR(tracking_timestamp,1,10) dt
                FROM    xyf_dwd.dwd_event_tracking_log_core_di
                WHERE   SUBSTR(TO_DATE(pt,"yyyyMMdd"),1,10) >= DATE_SUB(CURRENT_TIMESTAMP(),180)
                AND     tracking_id IN ('JK-11-0-0-241')
                AND     GET_JSON_OBJECT(biz_info,'$.hasFill') = 1
            ) t_bj ----风险补件
ON      t_bj.user_no = shouxin.user_no
AND     t_bj.dt = SUBSTR(shouxin.created_time,1,10)
LEFT JOIN   (
                SELECT  DISTINCT user_no
                        ,SUBSTR(tracking_timestamp,1,10) dt
                FROM    xyf_dwd.dwd_event_tracking_log_core_di
                WHERE   SUBSTR(TO_DATE(pt,"yyyyMMdd"),1,10) >= DATE_SUB(CURRENT_TIMESTAMP(),180)
                AND     (
                            (
                                        tracking_id = 'JK-66-0-0-1188'
                                        AND     GET_JSON_OBJECT(biz_info,'$.signDisplayStyle') IS NULL
                            )
                            OR      tracking_id = 'JK-51-0-0-1189'
                ) ---AND     tracking_id IN ('JK-66-0-0-1188','JK-51-0-0-1189')
            ) t_yzm ----验证码
ON      t_yzm.user_no = shouxin.user_no
AND     t_yzm.dt = SUBSTR(shouxin.created_time,1,10)
LEFT JOIN   (
                SELECT  DISTINCT user_no
                        ,SUBSTR(tracking_timestamp,1,10) dt
                FROM    xyf_dwd.dwd_event_tracking_log_core_di
                WHERE   SUBSTR(TO_DATE(pt,"yyyyMMdd"),1,10) >= DATE_SUB(CURRENT_TIMESTAMP(),180)
                AND     tracking_id IN ('JK-110-177-0-1303')
            ) t_qrtc ----确认弹窗
ON      t_qrtc.user_no = shouxin.user_no
AND     t_qrtc.dt = SUBSTR(shouxin.created_time,1,10)
LEFT JOIN   (
                SELECT  source
                        ,zero_channel
                        ,first_channel
                        ,second_channel
                FROM    xyf_bi_dev.utm_source_channel_v1_cdf_v
                WHERE   pt = MAX_PT('xyf_bi_dev.utm_source_channel_v1_cdf_v')
                GROUP BY source
                         ,zero_channel
                         ,first_channel
                         ,second_channel
            ) qu_gui
ON      qu_gui.source = c.attribution_source
LEFT JOIN   (
                SELECT  source
                        ,zero_channel
                        ,first_channel
                        ,second_channel
                FROM    xyf_bi_dev.utm_source_channel_v1_cdf_v
                WHERE   pt = MAX_PT('xyf_bi_dev.utm_source_channel_v1_cdf_v')
                GROUP BY source
                         ,zero_channel
                         ,first_channel
                         ,second_channel
            ) qu_zhu
ON      qu_zhu.source = c.current_utm_source
LEFT JOIN   (
                SELECT  *
                FROM    xyf_dws.dws_inloan_user_order_df
                WHERE   pt = '${bizdate}'
                AND     loan_flag = '首贷'
                AND     business_line IN ('APP','小程序端')
            ) o
ON      o.cust_no = shouxin.cust_no
LEFT JOIN   (
                SELECT                  ---order_number
                        ori_order_number
                        ,cust_no
                        ,CASE   WHEN y0_1_7 = 1 THEN due_amt / 100
                        END AS 出账7plus金额
                        ,NVL(CASE    WHEN y0_1_7 = 1 THEN y3_1_7 / 100 END,0) AS 出账7plus逾期金额
                        ,CASE   WHEN y0_1_15 = 1 THEN due_amt / 100
                        END AS 出账15plus金额
                        ,NVL(CASE    WHEN y0_1_15 = 1 THEN y3_1_15 / 100 END,0) AS 出账15plus逾期金额 --1_30
                        ,CASE   WHEN y0_1_30 = 1 THEN due_amt / 100
                        END AS 出账30plus金额
                        ,NVL(CASE    WHEN y0_1_30 = 1 THEN y3_1_30 / 100 END,0) AS 出账30plus逾期金额 --2_30
                        ,CASE   WHEN y0_2_30 = 1 THEN due_amt / 100
                        END AS 出账60plus金额
                        ,NVL(CASE    WHEN y0_2_30 = 1 THEN y3_2_30 / 100 END,0) AS 出账60plus逾期金额 --3_30
                        ,CASE   WHEN y0_3_30 = 1 THEN due_amt / 100
                        END AS 出账90plus金额
                        ,NVL(CASE    WHEN y0_3_30 = 1 THEN y3_3_30 / 100 END,0) AS 出账90plus逾期金额 --4_30
                        ,CASE   WHEN y0_4_30 = 1 THEN due_amt / 100
                        END AS 出账120plus金额
                        ,NVL(CASE    WHEN y0_4_30 = 1 THEN y3_4_30 / 100 END,0) AS 出账120plus逾期金额 --6_30
                        ,CASE   WHEN y0_5_30 = 1 THEN due_amt / 100
                        END AS 出账150plus金额
                        ,NVL(CASE    WHEN y0_5_30 = 1 THEN y3_5_30 / 100 END,0) AS 出账150plus逾期金额 --_30
                        ,CASE   WHEN y0_6_30 = 1 THEN due_amt / 100
                        END AS 出账180plus金额
                        ,NVL(CASE    WHEN y0_6_30 = 1 THEN y3_6_30 / 100 END,0) AS 出账180plus逾期金额
                        ,CASE   WHEN y0_7_30 = 1 THEN due_amt / 100
                        END AS 出账210plus金额
                        ,NVL(CASE    WHEN y0_7_30 = 1 THEN y3_7_30 / 100 END,0) AS 出账210plus逾期金额
                        ,CASE   WHEN y0_8_30 = 1 THEN due_amt / 100
                        END AS 出账240plus金额
                        ,NVL(CASE    WHEN y0_8_30 = 1 THEN y3_8_30 / 100 END,0) AS 出账240plus逾期金额
                        ,CASE   WHEN y0_9_30 = 1 THEN due_amt / 100
                        END AS 出账270plus金额
                        ,NVL(CASE    WHEN y0_9_30 = 1 THEN y3_9_30 / 100 END,0) AS 出账270plus逾期金额
                        ,CASE   WHEN y0_10_30 = 1 THEN due_amt / 100
                        END AS 出账300plus金额
                        ,NVL(CASE    WHEN y0_10_30 = 1 THEN y3_10_30 / 100 END,0) AS 出账300plus逾期金额
                        ,CASE   WHEN y0_11_30 = 1 THEN due_amt / 100
                        END AS 出账330plus金额
                        ,NVL(CASE    WHEN y0_11_30 = 1 THEN y3_11_30 / 100 END,0) AS 出账330plus逾期金额
                        ,CASE   WHEN y0_12_30 = 1 THEN due_amt / 100
                        END AS 出账360plus金额
                        ,NVL(CASE    WHEN y0_12_30 = 1 THEN y3_12_30 / 100 END,0) AS 出账360plus逾期金额 ---------
                        ,CASE   WHEN y0_1_7 = 1 THEN 1
                                ELSE 0
                        END AS fpd7_available
                        ,CASE   WHEN y0_1_15 = 1 THEN 1
                                ELSE 0
                        END AS fpd15_available
                        ,CASE   WHEN y0_1_30 = 1 THEN 1
                                ELSE 0
                        END AS m1_available
                        ,CASE   WHEN y0_2_30 = 1 THEN 1
                                ELSE 0
                        END AS m2_available
                        ,CASE   WHEN y0_3_30 = 1 THEN 1
                                ELSE 0
                        END AS m3_available
                        ,CASE   WHEN y0_4_30 = 1 THEN 1
                                ELSE 0
                        END AS m4_available
                        ,CASE   WHEN y0_5_30 = 1 THEN 1
                                ELSE 0
                        END AS m5_available
                        ,CASE   WHEN y0_6_30 = 1 THEN 1
                                ELSE 0
                        END AS m6_available
                        ,CASE   WHEN y0_7_30 = 1 THEN 1
                                ELSE 0
                        END AS m7_available
                        ,CASE   WHEN y0_8_30 = 1 THEN 1
                                ELSE 0
                        END AS m8_available
                        ,CASE   WHEN y0_9_30 = 1 THEN 1
                                ELSE 0
                        END AS m9_available
                        ,CASE   WHEN y0_10_30 = 1 THEN 1
                                ELSE 0
                        END AS m10_available
                        ,CASE   WHEN y0_11_30 = 1 THEN 1
                                ELSE 0
                        END AS m11_available
                        ,CASE   WHEN y0_12_30 = 1 THEN 1
                                ELSE 0
                        END AS m12_available ---FROM    xyf_dws.dws_repay_risk_order_bill_mob_df
                FROM    xyf_dws.dws_repay_risk_order_bill_mob_agg_df ---这个是大额拆单后的新表,区别在于，用户1笔拆成多笔之后有1笔违约就算全部违约。
                WHERE   pt = '${bizdate}'
            ) mob
ON      o.first_order_number = mob.ori_order_number
LEFT JOIN   (
                SELECT  *
                        ,GET_JSON_OBJECT(extend_data,'$.loanAmountFen') / 100 AS 预借款提交金额
                        ,GET_JSON_OBJECT(extend_data,'$.term') AS 预借款提交期数
                FROM    xyf_dwd.dwd_inloan_loan_pre_apply_df
                WHERE   pt = MAX_PT('xyf_dwd.dwd_inloan_loan_pre_apply_df')
            ) po
ON      po.cust_no = shouxin.cust_no
AND     o.first_order_number = po.relate_order_no
LEFT JOIN   (
                --是否在提现页买卡？
                SELECT  order_number_loan
                        ,order_status AS status
                        ,real_card_price
                        ,order_time
                        ,CASE   WHEN order_from IN ('lend-before',"lend_before_retain") THEN '借款页签约'
                        END AS is_借款页签约
                FROM    xyf_dwd.dwd_user_vip_order_df
                WHERE   pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
                AND     SUBSTR(order_time,1,10) >= '2024-08-01'
                AND     vip_card_type = 1 ----筛选会员卡购卡订单
                AND     order_status IN (1,3,4,5,6) ----成功签约，不一定成功付款
                AND     vip_order_number = first_vip_order_number --不看续约
                AND     order_time IS NOT NULL
                AND     app_user_id IS NOT NULL
                UNION ALL
                SELECT  loan_order_number AS order_number_loan
                        ,CASE   WHEN order_status = 'pay_success' THEN 3
                                ELSE 0
                        END AS status
                        ,real_card_price * 100 AS real_card_price ---因为上面的公式除了100，这里就先*100吧！
                        ,order_time
                        ,CASE   WHEN order_from LIKE '%lend_before%' THEN '借款页签约'
                                WHEN order_from LIKE '%lend_after%' THEN '卡单页签约'
                                ELSE '其他'
                        END AS is_借款页签约
                FROM    xyf_dwd.dwd_inloan_leap_vip_order_hf
                WHERE   pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
                AND     vip_order_number = first_vip_order_number --不看续约
                AND     order_time IS NOT NULL
                AND     SUBSTR(order_time,1,10) >= '2025-08-11'
            ) vip
ON      vip.order_number_loan = o.first_order_number
GROUP BY TO_DATE(wanjian.created_time)
         ,wt
         ,mt
         ,biz_type_bi
         ,biz_type_bi_3
         ,wanjian.user_no
         ,date(wanjian.created_time)
         ,is_API半流程
         ,is_虚假给额
         ,c.reg_channel
         ,phone_brand
         ,大盘随机数
         ,是否首申_风险
         ,是否首申_业务
         ,wanjian.created_time
;

DROP TABLE IF EXISTS xyf_bi_dev.bi_app_zhuanhua_shouxin_xujia_uid
;

CREATE TABLE xyf_bi_dev.bi_app_zhuanhua_shouxin_xujia_uid AS
SELECT  *
        ,CASE   WHEN create_cnt_dj = 0
                    AND jkfwxy_cnt = 1
                    AND fxsq_cnt = 0
                    AND vip_cnt = 0
                    AND bj_cnt = 0
                    AND yzm_cnt = 0
                    AND qrtc_cnt = 0 THEN 1
                ELSE 0
        END AS jkfwxy_lost_cnt
        ,CASE   WHEN create_cnt_dj = 0
                    AND fxsq_cnt = 1
                    AND vip_cnt = 0
                    AND bj_cnt = 0
                    AND yzm_cnt = 0
                    AND qrtc_cnt = 0 THEN 1
                ELSE 0
        END AS fxsq_lost_cnt
        ,CASE   WHEN create_cnt_dj = 0
                    AND vip_cnt = 1
                    AND bj_cnt = 0
                    AND yzm_cnt = 0
                    AND qrtc_cnt = 0 THEN 1
                ELSE 0
        END AS vip_lost_cnt
        ,CASE   WHEN create_cnt_dj = 0
                    AND bj_cnt = 1
                    AND yzm_cnt = 0
                    AND qrtc_cnt = 0 THEN 1
                ELSE 0
        END AS bj_lost_cnt
        ,CASE   WHEN create_cnt_dj = 0
                    AND yzm_cnt = 1
                    AND qrtc_cnt = 0 THEN 1
                ELSE 0
        END AS yzm_lost_cnt
        ,CASE   WHEN create_cnt_dj = 0 AND qrtc_cnt = 1 -- AND jkfwxy_cnt = 0
                -- AND fxsq_cnt = 0
                -- AND vip_cnt = 0
                -- AND bj_cnt = 0
                -- AND yzm_cnt = 0 
                THEN 1
                ELSE 0
        END AS qrtc_lost_cnt
        ,CASE   WHEN 登录版本 >= '60009' THEN CONCAT_WS('.',SUBSTR(登录版本,1,1),SUBSTR(登录版本,3,1),SUBSTR(登录版本,5,2))
                WHEN 登录版本 < '60009' THEN '6.0.8及以下'
                ELSE 'other'
        END AS app_login_version
FROM    xyf_bi_dev.bi_app_zhuanhua_shouxin_xujia_uid_T0
;

DROP TABLE IF EXISTS xyf_bi_dev.bi_app_zhuanhua_shouxin_xujia
;

CREATE TABLE xyf_bi_dev.bi_app_zhuanhua_shouxin_xujia AS
SELECT  dt
        ,wt
        ,mt
        ,授信金额_level
        ,biz_type_bi
        ,biz_type_bi_3
        ,CASE   WHEN 实际执行人行完件评级_v3_v4_xifen >= 8 THEN '8+'
                WHEN 实际执行人行完件评级_v3_v4_xifen > 0 AND 实际执行人行完件评级_v3_v4_xifen <= 2 THEN 1
                WHEN 实际执行人行完件评级_v3_v4_xifen > 2 AND 实际执行人行完件评级_v3_v4_xifen < 8 THEN FLOOR(实际执行人行完件评级_v3_v4_xifen)
                ELSE ''
        END AS 实际执行人行完件评级_v3_v4
        ,实际执行人行完件评级_v3_v4_xifen
        ,注册授信时间差
        ,CASE   WHEN API虚假给额 = 1 THEN 'API虚假给额'
                when 半流程虚假给额 = 1 THEN '半流程虚假给额'
                WHEN is_虚假给额 = '虚假给额'  THEN 'APP虚假给额'
        END AS 是否API虚假给额
        ,is_API半流程
        ,is_虚假给额
        ,是否首申_风险
        ,是否首申_业务
        ,CASE   WHEN dt >= DATE_SUB(CURRENT_TIMESTAMP(),60) THEN phone_brand
                ELSE '仅保留最近60天数据'
        END AS phone_brand
        ,CASE   WHEN dt >= DATE_SUB(CURRENT_TIMESTAMP(),60) THEN app_login_version
                ELSE '仅保留最近60天数据'
        END AS app_login_version
        ,大盘随机数
        ,CASE   WHEN 是否通过预借款发起 = 1 THEN '预借款发起'
                ELSE '否'
        END AS 是否预借款
        ,CASE   WHEN dt >= DATE_SUB(CURRENT_TIMESTAMP(),60) THEN 注册渠道_产品口径
                ELSE '仅保留最近60天数据'
        END AS 注册渠道_产品口径
        ,COUNT(DISTINCT user_no) 完件人数
        ,COUNT(DISTINCT CASE    WHEN 授信额度_shu > 0 THEN user_no END) 授信通过
        ,SUM(授信额度_shu) 授信额度_shu
        ,SUM(提现人) 提现人
        ,SUM(T7提现人) T7提现人
        ,SUM(T30提现人) T30提现人
        ,SUM(提现金额) 提现金额
        ,SUM(T7提现金额) T7提现金额
        ,SUM(T30提现金额) T30提现金额
        ,SUM(累积提现金额) 累积提现金额
        ,SUM(提现金额_period) 提现金额_period
        ,SUM(T0风险通过_提现金额) T0风险通过_提现金额
        ,SUM(T7风险通过_提现金额) T7风险通过_提现金额
        ,SUM(放款金额) 放款金额
        ,SUM(T7放款金额) T7放款金额
        ,SUM(T30放款金额) T30放款金额
        ,SUM(放款金额_period) 放款金额_period
        ,SUM(风控通过人) 风控通过人
        ,SUM(T7风控通过人) T7风控通过人
        ,SUM(放款人) 放款人
        ,SUM(t7放款人) t7放款人
        ,SUM(t30放款人) t30放款人
        ,SUM(授信额度_放款) 授信额度_放款
        ,SUM(授信额度_放款7) 授信额度_放款7
        ,SUM(授信额度_放款30) 授信额度_放款30
        ,SUM(授信额度_放款累积) 授信额度_放款累积
        ,SUM(T0会员卡签约) T0会员卡签约
        ,SUM(T0会员卡扣款) T0会员卡扣款
        ,SUM(T0会员卡扣款金额) T0会员卡扣款金额
        ,SUM(T7会员卡签约) T7会员卡签约
        ,SUM(T7会员卡扣款) T7会员卡扣款
        ,SUM(T7会员卡扣款金额) T7会员卡扣款金额
        ,SUM(T30会员卡签约) T30会员卡签约
        ,SUM(T30会员卡扣款) T30会员卡扣款
        ,SUM(T30会员卡扣款金额) T30会员卡扣款金额
        ,SUM(累积会员卡签约) 累积会员卡签约
        ,SUM(累积会员卡扣款) 累积会员卡扣款
        ,SUM(累积会员卡扣款金额) 累积会员卡扣款金额
        ,SUM(累积放款金额) 累积放款金额
        ,SUM(累积提现人) 累积提现人
        ,SUM(累积放款人) 累积放款人
        ,SUM(累积放款金额_定价) 累积放款金额_定价
        ,SUM(出账7plus金额) 出账7plus金额
        ,SUM(出账15plus金额) 出账15plus金额
        ,SUM(出账30plus金额) 出账30plus金额
        ,SUM(出账60plus金额) 出账60plus金额
        ,SUM(出账90plus金额) 出账90plus金额
        ,SUM(出账120plus金额) 出账120plus金额
        ,SUM(出账150plus金额) 出账150plus金额
        ,SUM(出账180plus金额) 出账180plus金额
        ,SUM(出账210plus金额) 出账210plus金额
        ,SUM(出账240plus金额) 出账240plus金额
        ,SUM(出账270plus金额) 出账270plus金额
        ,SUM(出账300plus金额) 出账300plus金额
        ,SUM(出账330plus金额) 出账330plus金额
        ,SUM(出账360plus金额) 出账360plus金额
        ,SUM(出账7plus逾期金额) 出账7plus逾期金额
        ,SUM(出账15plus逾期金额) 出账15plus逾期金额
        ,SUM(出账30plus逾期金额) 出账30plus逾期金额
        ,SUM(出账60plus逾期金额) 出账60plus逾期金额
        ,SUM(出账90plus逾期金额) 出账90plus逾期金额
        ,SUM(出账120plus逾期金额) 出账120plus逾期金额
        ,SUM(出账150plus逾期金额) 出账150plus逾期金额
        ,SUM(出账180plus逾期金额) 出账180plus逾期金额
        ,SUM(出账210plus逾期金额) 出账210plus逾期金额
        ,SUM(出账240plus逾期金额) 出账240plus逾期金额
        ,SUM(出账270plus逾期金额) 出账270plus逾期金额
        ,SUM(出账300plus逾期金额) 出账300plus逾期金额
        ,SUM(出账330plus逾期金额) 出账330plus逾期金额
        ,SUM(出账360plus逾期金额) 出账360plus逾期金额
        ,SUM(fpd7_available) fpd7_available
        ,SUM(fpd15_available) fpd15_available
        ,SUM(m1_available) m1_available
        ,SUM(m2_available) m2_available
        ,SUM(m3_available) m3_available
        ,SUM(m4_available) m4_available
        ,SUM(m5_available) m5_available
        ,SUM(m6_available) m6_available
        ,SUM(m7_available) m7_available
        ,SUM(m8_available) m8_available
        ,SUM(m9_available) m9_available
        ,SUM(m10_available) m10_available
        ,SUM(m11_available) m11_available
        ,SUM(m12_available) m12_available
        ,SUM(back_cnt) back_cnt
        ,SUM(提现页_cnt) 提现页_cnt
        ,SUM(点击借款按钮_cnt) 点击借款按钮_cnt
        ,SUM(jkfwxy_cnt) jkfwxy_cnt
        ,SUM(fxsq_cnt) fxsq_cnt
        ,SUM(vip_cnt) vip_cnt
        ,SUM(bj_cnt) bj_cnt
        ,SUM(yzm_cnt) yzm_cnt
        ,SUM(qrtc_cnt) qrtc_cnt
        ,SUM(jkfwxy_lost_cnt) jkfwxy_lost_cnt
        ,SUM(fxsq_lost_cnt) fxsq_lost_cnt
        ,SUM(vip_lost_cnt) vip_lost_cnt
        ,SUM(bj_lost_cnt) bj_lost_cnt
        ,SUM(yzm_lost_cnt) yzm_lost_cnt
        ,SUM(qrtc_lost_cnt) qrtc_lost_cnt ---back_cnt	提现页_cnt	点击借款按钮_cnt	create_cnt_dj	jkfwxy_cnt	fxsq_cnt	vip_cnt	bj_cnt	yzm_cnt	qrtc_cnt	jkfwxy_lost_cnt	fxsq_lost_cnt	vip_lost_cnt	bj_lost_cnt	yzm_lost_cnt	qrtc_lost_cnt
FROM    xyf_bi_dev.bi_app_zhuanhua_shouxin_xujia_uid
GROUP BY dt
         ,wt
         ,mt
         ,授信金额_level
         ,biz_type_bi
         ,biz_type_bi_3
         ,实际执行人行完件评级_v3_v4
         ,实际执行人行完件评级_v3_v4_xifen
         ,注册授信时间差
         ,是否API虚假给额
         ,is_API半流程
         ,is_虚假给额
         ,是否预借款
         ,注册渠道_产品口径
         ,phone_brand
         ,app_login_version
         ,大盘随机数
         ,是否首申_风险
         ,是否首申_业务
;
-- """
-- SELECT
--   date(shouxin.row_crt_ts) dt,
-- 	zhou.week_range wt,
--     substr( shouxin.row_crt_ts,1,7 ) mt,
-- 	-- 授信额度  
-- 	 case when    shouxin.shouxin_cnt=1 and 0<=g.final_amt and g.final_amt <=3000 then '1. 0k-3k'
-- 	      when shouxin.shouxin_cnt=1 and 3000<g.final_amt and g.final_amt <=5000 then '2. 3k-5k'
--                   when   shouxin.shouxin_cnt=1 and 5000<g.final_amt and g.final_amt<=10000 then '3. 5k-1w'
--                   when   shouxin.shouxin_cnt=1 and 10000<g.final_amt and g.final_amt<=20000 then '4. 1w-2w'
--                   when    shouxin.shouxin_cnt=1 and 20000<g.final_amt  and g.final_amt <=50000   then '5. 2w-5w'
--                   when    shouxin.shouxin_cnt=1 and 50000<g.final_amt   then '6. 5w+'
--                   else ''
--             end as 授信金额_level,
-- --  注册渠道
-- --    case 
-- --     when lower(c.current_utm_source)   like '%api%' then 'API' 
-- --        when c.current_utm_source LIKE 'QD-XXL-HW%' THEN 'app其他' 
-- --        when c.current_utm_source  LIKE '%-XXL-%' THEN '信息流' 
-- --        when c.current_utm_source   like '%-DX-%' OR  c.current_utm_source   like '%DX-XYF01-TPKJ%'  OR  c.current_utm_source   like '%腾云天下%'  then '短信' 
-- -- else 'app其他' end  as  biz_type_bi,
-- qu_zhu.first_channel as  biz_type_bi,
-- -- 归因渠道
-- -- case 
-- --     when lower(c.attribution_source)   like '%api%' then 'API' 
-- --        when c.attribution_source LIKE 'QD-XXL-HW%' THEN 'app其他' 
-- --        when c.attribution_source  LIKE '%-XXL-%' THEN '信息流' 
-- --        when c.attribution_source   like '%-DX-%' OR  c.attribution_source   like '%DX-XYF01-TPKJ%'  OR  c.attribution_source  like '%腾云天下%'  then '短信' 
-- -- else 'app其他' end  as  biz_type_bi_3,
-- qu_gui.first_channel  as  biz_type_bi_3,
-- -- 完件评级
-- v1_score as 人行完件评级V1版本,
-- ping2.score as 人行评级V3,
-- -- v2.personalloan_sd_line_amtlevel_rh_app as 实际执行人行完件评级_V3_V4,
-- v2.personalloan_sd_line_amtlevel_rh_app as 实际执行人行完件评级_V3_V4_xifen,
--  case when v2.personalloan_sd_line_amtlevel_rh_app>0 and v2.personalloan_sd_line_amtlevel_rh_app<2 then 1
--       when v2.personalloan_sd_line_amtlevel_rh_app>=2 and v2.personalloan_sd_line_amtlevel_rh_app<3 then 2
--       when v2.personalloan_sd_line_amtlevel_rh_app>=3 and v2.personalloan_sd_line_amtlevel_rh_app<4 then 3
--       when v2.personalloan_sd_line_amtlevel_rh_app>=4 and v2.personalloan_sd_line_amtlevel_rh_app<5 then 4
--       when v2.personalloan_sd_line_amtlevel_rh_app>=5 and v2.personalloan_sd_line_amtlevel_rh_app<6 then 5
--       when v2.personalloan_sd_line_amtlevel_rh_app>=6 and v2.personalloan_sd_line_amtlevel_rh_app<7 then 6
--       when v2.personalloan_sd_line_amtlevel_rh_app>=7 and v2.personalloan_sd_line_amtlevel_rh_app<8 then 7
--       when v2.personalloan_sd_line_amtlevel_rh_app>=8  then '8+'
--       else ''
--  end as 实际执行人行完件评级_V3_V4,
-- shouxin.shouxin_type as 是否API授信,
-- shouxin.yewu_type as 个人_现金,
-- case when datediff(shouxin.row_crt_ts,c.created_time) =0 then '0d'
-- 	when 0<datediff(shouxin.row_crt_ts,c.created_time) and datediff(shouxin.row_crt_ts,c.created_time)<=30 then '1-30d'
--     when datediff(shouxin.row_crt_ts,c.created_time)>30 and datediff(shouxin.row_crt_ts,c.created_time)<=60 then '31d-60d'
-- 	when datediff(shouxin.row_crt_ts,c.created_time)>60 then '61d+'
-- 	 else '' 
--      end as 注册授信时间差,
--      case when  c.current_utm_source   like '%-DX-%' OR  c.current_utm_source   like '%DX-XYF01-TPKJ%'  OR  c.current_utm_source   like '%腾云天下%'   then n.name   else  '' end as 短信渠道商,
-- count(distinct case when    shouxin.shouxin_cnt=1 then shouxin.mobile end ) 授信通过,
-- sum(case when    shouxin.shouxin_cnt=1 then g.final_amt  else 0 end )  授信额度_shu,
-- count(distinct case when datediff(d.create_date,shouxin.row_crt_ts)=0 and  d.apply_cnt=1 then shouxin.mobile end ) 提现人,
-- count(distinct case when datediff(d.create_date,shouxin.row_crt_ts)>=0 and datediff(d.create_date,shouxin.row_crt_ts)<=3 AND d.apply_cnt=1 then shouxin.mobile end ) T3提现人,
-- count(distinct case when datediff(d.create_date,shouxin.row_crt_ts)>=0 and datediff(d.create_date,shouxin.row_crt_ts)<=7 AND d.apply_cnt=1 then shouxin.mobile end ) T7提现人,
-- count(distinct case when datediff(d.create_date,shouxin.row_crt_ts)>=0 and datediff(d.create_date,shouxin.row_crt_ts)<=30 AND d.apply_cnt=1 then shouxin.mobile end ) T30提现人,
-- count( case when datediff(d.create_date,shouxin.row_crt_ts)=0 and  d.apply_cnt=1 then shouxin.mobile end ) 提现人次,
-- -- 体现期限
-- sum( case when datediff(d.create_date,shouxin.row_crt_ts)=0 then   d.apply_amt else 0 end ) 提现金额,
-- sum( case when datediff(d.create_date,shouxin.row_crt_ts)=0 then   d.apply_amt*D.apply_PERIOD else 0 end ) 提现金额_PERIOD,
-- -- 放款期限
-- sum(  case when datediff(d.pay_DATE,shouxin.row_crt_ts)=0 then   d.loan_amt else 0 end ) 放款金额,
-- sum(  case when datediff(d.pay_DATE,shouxin.row_crt_ts)>=0 AND  datediff(d.pay_DATE,shouxin.row_crt_ts)<=3  then   d.loan_amt else 0 end ) T3放款金额,
-- sum(  case when datediff(d.pay_DATE,shouxin.row_crt_ts)>=0 AND  datediff(d.pay_DATE,shouxin.row_crt_ts)<=7  then   d.loan_amt else 0 end ) T7放款金额,
-- sum(  case when datediff(d.pay_DATE,shouxin.row_crt_ts)=0 then   d.loan_amt*d.PERIOD else 0 end ) 放款金额_PERIOD,
-- count(distinct case when datediff(d.create_date,shouxin.row_crt_ts)=0 and  d.risk_pass=1 then shouxin.mobile end ) 风控通过人,
-- count(distinct case when datediff(d.create_date,shouxin.row_crt_ts)>=0 and datediff(d.create_date,shouxin.row_crt_ts)<=3 and  d.risk_pass=1 then shouxin.mobile end ) T3风控通过人,
-- count(distinct case when datediff(d.create_date,shouxin.row_crt_ts)>=0 and datediff(d.create_date,shouxin.row_crt_ts)<=7 and  d.risk_pass=1 then shouxin.mobile end ) T7风控通过人,
-- count(distinct case when datediff(d.pay_DATE,shouxin.row_crt_ts)=0 and  d.remit_pass=1 then shouxin.mobile end ) 资方通过人,
-- count(distinct case when datediff(d.pay_DATE,shouxin.row_crt_ts)>=0 and datediff(d.pay_DATE,shouxin.row_crt_ts)<=3 and  d.remit_pass=1 then shouxin.mobile end ) T3资方通过人,
-- count(distinct case when datediff(d.pay_DATE,shouxin.row_crt_ts)>=0 and datediff(d.pay_DATE,shouxin.row_crt_ts)<=7 and  d.remit_pass=1 then shouxin.mobile end ) T7资方通过人,
-- count(distinct case when datediff(d.pay_DATE,shouxin.row_crt_ts)=0 and  d.loan_cnt=1 then shouxin.mobile end ) 放款人,
-- count(distinct case when datediff(d.pay_DATE,shouxin.row_crt_ts)>=0 and datediff(d.pay_DATE,shouxin.row_crt_ts)<=3 and d.loan_cnt=1 then shouxin.mobile end ) T3放款人,
-- count(distinct case when datediff(d.pay_DATE,shouxin.row_crt_ts)>=0 and datediff(d.pay_DATE,shouxin.row_crt_ts)<=7 and d.loan_cnt=1 then shouxin.mobile end ) T7放款人,
-- count(distinct case when datediff(d.pay_DATE,shouxin.row_crt_ts)>=0 and datediff(d.pay_DATE,shouxin.row_crt_ts)<=30 and d.loan_cnt=1 then shouxin.mobile end ) T30放款人,
-- sum(  case when datediff(d.pay_DATE,shouxin.row_crt_ts)=0  and shouxin.shouxin_cnt=1 then g.final_amt   else 0 end ) 授信额度_放款,
--     sum(  case when (datediff(d.pay_DATE,shouxin.row_crt_ts) between 0 and 7 )  and shouxin.shouxin_cnt=1 then g.final_amt   else 0 end ) 授信额度_放款7,
--       sum(  case when (datediff(d.pay_DATE,shouxin.row_crt_ts) between 0 and 30  ) and shouxin.shouxin_cnt=1 then g.final_amt   else 0 end ) 授信额度_放款30
-- FROM
-- (
-- --     select id_card_number
-- --                      ,mobile
-- --                          ,app
-- --                          ,date(row_crt_ts) as row_crt_ts_date
-- --                          ,case when lower(inner_app) in ('xyf01','fxk','cxh') then "app" else "api" end as shouxin_type 
-- --                          ,case when lower(inner_app) in ('xyf01','fxk') then "个人" 
-- --                                when lower(inner_app) ='cxh' then "现金"  else "" end as yewu_type     
-- --                                 ,cust_no	
-- --                          ,max(case when activation_status='success' then 1 else 0 end)  as shouxin_cnt
-- --                     ,max(row_crt_ts) as row_crt_ts
-- --                         --  from xyf_ods.ods_xyf_bi_cash_activation_log_df
-- --                          from xyf_dwd.dwd_xyf_bi_cash_activation_log_df_v
-- --                          where  pt = '${bizdate}' and app in ('xyf01','fxk') and lower(inner_app) in ('xyf01','fxk') and activation_status='success'  and activation_source !='fxk_copy'
-- -- ;
--      select id_card_number
--                         ,mobile
--                         ,app
--                         ,date(created_time) as row_crt_ts_date
--                         ,case when lower(inner_app) in ('xyf01','fxk','cxh') then "app" else "api" end as shouxin_type 
--                         ,"个人"  as yewu_type 
--                         ,cust_no
--                         --,1 as wanjian_cnt
--                         ,case when status = 2 then 1 else 0 end as shouxin_cnt
--                         ,created_time as row_crt_ts
--                          FROM    xyf_dwd.dwd_preloan_credit_apply_df
--                             WHERE   pt = '${bizdate}'
--                             --AND     date(created_time) >= '2024-01-01'
--                             AND     app IN ('xyf01') --
--                             AND     status = 2 --成功
--                             AND     inner_app = 'xyf01'
--                          and biz_flow_number  in (
--  -- 虚假给额的授信成功用户口径，biz_flow_number关联授信表
-- SELECT  biz_flow_number -- 授信biz_flow_number
--         -- ,id_card_number
--         -- ,GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") AS 人群类型
-- FROM    xyf_dwd.dwd_inloan_t_decision_result_detail_df
-- WHERE   pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
-- AND     enginecode = 'jcl_20240923000003'
-- -- AND     GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") = 'fake_activation'
-- and GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") rlike 'fake_activation'
-- AND     decision_time >= '2024-10-15 00:00:00' --
-- -- 存在少量异常数据，同一个biz_flow_number+enginename 存在多条记录；下面排序做个兜底的清洗
-- QUALIFY ROW_NUMBER() OVER (PARTITION BY biz_flow_number,date(decision_time) ORDER BY decision_time DESC ) = 1
--                     )
--                     QUALIFY row_number() OVER (PARTITION BY cust_no order by created_time DESC ) = 1
--                     --      group by 
--                     --      id_card_number
--                     --  ,mobile
--                     --      ,app
--                     --      ,date(row_crt_ts) 
--                     --      ,case when lower(inner_app) in ('xyf01','fxk','cxh') then "app" else "api" end 
--                     --      ,case when lower(inner_app) in ('xyf01','fxk') then "个人" 
--                     --            when lower(inner_app) ='cxh' then "现金"  else "" end 
--                     --      ,cust_no	
--                                 ) shouxin 
-- -- left join (
-- -- select c.*, case when c2.attribution_source is null then c.current_utm_source else  c2.attribution_source  end as attribution_source  
-- --  ,SUBSTR(conv2(sha2(CONCAT('xinkeyunying',c.id),256),16,10),-2) as rand_num
-- -- from 
-- --         xyf_ods.ods_xyf_bi_credit_user_df c
-- --         left join (SELECT * FROM 
-- --             (SELECT 
-- --                 user_id,
-- --                 attribution_source,
-- --                 ROW_NUMBER() over(partition by user_id order by created_time desc) as rn
-- --             FROM
-- --                 xyf_ods.ods_xyf_flow_sys_flow_attribution_result_df
-- --             where
-- --                 pt = '${bizdate}')
-- --         WHERE  rn = 1) c2 on c.id = c2.user_id
-- --        where
-- --         c.pt = '${bizdate}'
-- --         and  c.channel  not in ('fxk_copy' )
-- --         -- and c.app  in ('xyf01')
-- --         ) c on c.mobile =shouxin.mobile and c.app =shouxin.app
-- left join (
-- select c.*, case when c2.attribution_source is null then c.current_utm_source else  c2.attribution_source  end as attribution_source  
--  ,SUBSTR(conv2(sha2(CONCAT('xinkeyunying',c.id),256),16,10),-2) as rand_num
-- from 
--         (select *, app_user_id as id, register_time as created_time  from xyf_dim.dim_user_app_basic_info_df   where     pt = '${bizdate}' )c
--         left join (SELECT * FROM 
--             (SELECT 
--                 user_id,
--                 attribution_source,
--                 ROW_NUMBER() over(partition by user_id order by created_time desc) as rn
--             FROM
--                 xyf_dwd.dwd_xyf_flow_sys_flow_attribution_result_dup_df
--             where
--                 pt = '${bizdate}')
--         WHERE  rn = 1) c2 on c.id = c2.user_id
--        where
--         c.pt = '${bizdate}'
--         and  c.channel  not in ('fxk_copy' )
--         -- and c.app  in ('xyf01')
--         ) c on c.mobile =shouxin.mobile and c.app =shouxin.app
--           left join (
--          select source,zero_channel,first_channel,second_channel
-- from xyf_bi_dev.utm_source_channel_v1_cdf_v
-- where pt = max_pt('xyf_bi_dev.utm_source_channel_v1_cdf_v')
-- group by source,zero_channel,first_channel,second_channel
--      )  qu_gui on qu_gui.source =c.attribution_source
--         left join (
--          select source,zero_channel,first_channel,second_channel
-- from xyf_bi_dev.utm_source_channel_v1_cdf_v
-- where pt = max_pt('xyf_bi_dev.utm_source_channel_v1_cdf_v')
-- group by source,zero_channel,first_channel,second_channel
--      )  qu_zhu on qu_zhu.source =c.current_utm_source
--         left join (
-- 		SELECT 
--         abbr,
--         name
--     FROM
--         -- xyf_ods.ods_sfy_sta_channel_df
--         xyf_dwd.dwd_sfy_sta_channel_df_v
--     -- WHERE 
--     --     pt = MAX_PT('xyf_ods.ods_sfy_sta_channel_df')
--     GROUP BY 
--         abbr,
--         name
-- ) n on c.current_utm_source=n.abbr
--    LEFT JOIN 
--     (SELECT 
--         mobile,
--         personalloan_sd_line_amtlevel_rh_app_v3_1_0 as score,
--         ROW_NUMBER() over(PARTITION BY mobile order by decision_time desc) as rn
--     FROM
--     (
--     --    select decision_time, mobile, biz_flow_number ,personalloan_sd_line_amtlevel_rh_app_v3_1_0 from xyf_ods.ods_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_1_0_v2_df
--     --    WHERE 
--     --     pt = '${bizdate}' and decision_time<'2024-01-05 15:58:11'
--         select decision_time, mobile, biz_flow_number ,personalloan_sd_line_amtlevel_rh_app_v3_1_0 from xyf_dim.dim_backup_ods_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_1_0_v2_df
--        WHERE 
--         pt = MAX_PT("xyf_dim.dim_backup_ods_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_1_0_v2_df") and decision_time<'2024-01-05 15:58:11'
--        union all
--     --    select decision_time, mobile, biz_flow_number ,personalloan_sd_line_amtlevel_rh_app_v3_3_0 as personalloan_sd_line_amtlevel_rh_app_v3_1_0 from xyf_dwd.dwd_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_3_0_v2_df_v 
--     --    WHERE 
--     --     pt = '${bizdate}'
--      select decision_time, mobile, biz_flow_number ,personalloan_sd_line_amtlevel_rh_app_v3_3_0 as personalloan_sd_line_amtlevel_rh_app_v3_1_0 
--     from 
--     xyf_dim.dim_backup_ods_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_3_0_v2_df
--        WHERE 
--         pt = MAX_PT("xyf_dim.dim_backup_ods_xyf_model_personalloan_sd_line_amtlevel_rh_app_v3_3_0_v2_df")
--     )
--     ) ping2 on shouxin.mobile = ping2.mobile and ping2.rn = 1
-- --     left join (
-- --     select mobile, strategy ,personalloan_sd_line_amtlevel_rh_app from 
-- --     (select mobile,strategy,personalloan_sd_line_amtlevel_rh_app, ROW_NUMBER() over(partition by mobile order by row_crt_ts desc) as rn  from xyf_fengkong.personalloan_sd_line_amtlevel_rh_app_v4_a_score_apply WHERE   app = 'xyf01') 
-- --     WHERE rn = 1
-- -- ) V2 on v2.mobile=shouxin.mobile   
-- left join (
--     select mobile, strategy ,personalloan_sd_line_amtlevel_rh_app from 
--     (select mobile,strategy,personalloan_sd_line_amtlevel_rh_app, ROW_NUMBER() over(partition by mobile order by row_crt_ts desc) as rn  from xyf_fengkong.sd_app_activation_amtlevel WHERE   app = 'xyf01' and activation_status in ('success')) 
--     WHERE rn = 1
-- ) V2 on v2.mobile=shouxin.mobile  
-- -- 完件评级
--     -- LEFT JOIN 
--     -- (SELECT 
--     --     mobile,
--     --     personalloan_sd_line_amtlevel_rh_app_v1 as v1_score,
--     --     ROW_NUMBER() over(PARTITION BY mobile order by decision_time desc) as rn
--     -- FROM
--     --     xyf_ods.ods_xinyongfei_model_personalloan_sd_line_amtlevel_rh_app_v1_v2_df
--     -- WHERE 
--     --     pt = '${bizdate}') ping on shouxin.mobile = ping.mobile and ping.rn = 1
--          LEFT JOIN 
--     (SELECT 
--         mobile,
--         personalloan_sd_line_amtlevel_rh_app_v1 as v1_score,
--         ROW_NUMBER() over(PARTITION BY mobile order by decision_time desc) as rn
--     FROM
--         -- xyf_ods.ods_xinyongfei_model_personalloan_sd_line_amtlevel_rh_app_v1_v2_df
--         xyf_dim.dim_backup_ods_xinyongfei_model_personalloan_sd_line_amtlevel_rh_app_v1_v2_df
--     WHERE 
--         pt = max_pt('xyf_dim.dim_backup_ods_xinyongfei_model_personalloan_sd_line_amtlevel_rh_app_v1_v2_df')) ping  on shouxin.mobile = ping.mobile and ping.rn = 1
--     --    LEFT JOIN   (
--     --                         SELECT  app_user_id
--     --                                 -- ,date(first_order_time) AS draw_apply_time
--     --                                 ,case when lower(inner_app) in ('xyf01','fxk','cxh') then "app" else "api" end as loan_type
--     --                                 ,mobile
--     --                                 ,app
--     --                                 ,min(first_order_time) as create_date
--     --                                 ,min(loan_time) as PAY_date
--     --                                 ,max(CASE    WHEN id_card_number is not null  THEN 1 else 0 END) AS apply_cnt
--     --                                 ,max(case when risk_status = "pass" then 1 else 0 end) as risk_pass  -- 风控通过
--     --                                 ,max(case when risk_status = "pass" and loan_status = "success" then 1 else 0 end) as remit_pass  -- 资金通过
--     --                                 ,max(case when id_card_number is not null then amount/100 else 0 end) as apply_amt  -- 提现金额
--     --                                 ,max(case when id_card_number is not null then period else 0 end) as apply_PERIOD  -- 提现期限
--     --                                 ,max(case when loan_time is not null then 1 else 0 end) as loan_cnt  -- 放款人
--     --                                 ,max(case when loan_time is not null then amount/100 else 0 end) as loan_amt
--     --                                 ,max(case when  loan_time is not null then period else 0 end) as PERIOD
--     --                         FROM    (
--     --                                     SELECT  *
--     --                                             ,ROW_NUMBER() OVER (PARTITION BY first_order_number ORDER BY order_route DESC ) AS rn
--     --                                     FROM    xyf_dwd.dwd_user_order_info_df
--     --                                     WHERE   pt = '${bizdate}'
--     --                                      AND     app IN ('xyf01')
--     --                                    -- AND     date(first_order_time) >= '2024-01-01'
--     --                                        and lower(inner_app) = 'xyf01' 
--     --                                      AND     loan_type_flag = '首贷'
--     --                                 ) 
--     --                         WHERE   rn = 1
--     --                         GROUP BY app_user_id
--     --                                 -- ,date(first_order_time) 
--     --                                 ,case when lower(inner_app) in ('xyf01','fxk','cxh') then "app" else "api" end 
--     --                                 ,mobile
--     --                                 ,app
--     --                     ) d
--     --         on shouxin.mobile =d.mobile and  shouxin.app =d.app  -- 考虑用户多次申请授信，只匹授信成功的
-- LEFT JOIN   (
--                             SELECT  user_no AS  app_user_id
--                                     -- ,date(first_order_time) AS draw_apply_time
--                                     ,case when lower(inner_app) in ('xyf01','fxk','cxh') then "app" else "api" end as loan_type
--                                    ,cust_no
--                                     ,app
--                                     ,min(first_order_time) as create_date
--                                     ,min(loan_time) as PAY_date
--                                     ,max(CASE    WHEN first_order_time IS NOT NULL  THEN 1 else 0 END) AS apply_cnt
--                                     ,max(case when risk_status = "pass" then 1 else 0 end) as risk_pass  -- 风控通过
--                                     ,max(case when risk_status = "pass" and loan_status = "success" then 1 else 0 end) as remit_pass  -- 资金通过
--                                     ,max(case when first_order_time IS NOT NULL  then loan_amt else 0 end) as apply_amt  -- 提现金额
--                                     ,max(case when first_order_time IS NOT NULL  then period else 0 end) as apply_PERIOD  -- 提现期限
--                                     ,max(case when loan_status = "success" then 1 else 0 end) as loan_cnt  -- 放款人
--                                     ,max(case when loan_status = "success" then loan_amt else 0 end) as loan_amt
--                                     ,max(case when  loan_status = "success" then period else 0 end) as PERIOD
--                             FROM    
--                                xyf_dws.dws_inloan_user_order_df
--                                         WHERE   pt = '${bizdate}'
--                                          AND     app IN ('xyf01')
--                                        -- AND    date(first_order_time) >= '2024-01-01'
--                                            and lower(inner_app) = 'xyf01' 
--                                          AND     loan_flag = '首贷'
--                             GROUP BY user_no 
--                                     -- ,date(first_order_time) 
--                                     ,case when lower(inner_app) in ('xyf01','fxk','cxh') then "app" else "api" end 
--                                    ,cust_no
--                                     ,app
--                         ) d
--             on shouxin.cust_no =d.cust_no and  shouxin.app =d.app  -- 考虑用户多次申请授信，只匹授信成功的
-- left join (
-- --     select  id_card_number,app,
-- --                 ROW_NUMBER() OVER ( partition by id_card_number,app order by created_time ) as rank_no
-- --            ,final_amt / 100 as final_amt
-- -- --            ,date(max(created_time)) as decision_time
-- --         --    from xyf_ods.ods_xyf_bi_amount_manage_log_2_df 
-- --             from xyf_dwd.dwd_xyf_bi_amount_manage_log_2_df
-- --            where pt = '${bizdate}' and  manage_type = 'cash_activation'
-- --            -- 额度激活 授信成功的额度
-- --            and app in ('fxk', 'xyf01')
-- --            ;
--     select id_card_number
--                         ,mobile
--                         ,app
--                         ,date(created_time) as row_crt_ts_date
--                         ,case when lower(inner_app) in ('xyf01','fxk','cxh') then "app" else "api" end as shouxin_type 
--                         ,"个人"  as yewu_type 
--                         ,cust_no
--                         --,1 as wanjian_cnt
--                         ,case when status = 2 then 1 else 0 end as shouxin_cnt
--                         ,created_time as row_crt_ts
--                         ,init_credit_line/100 as final_amt
--                          FROM    xyf_dwd.dwd_preloan_credit_apply_df
--                             WHERE   pt = '${bizdate}'
--                             --AND     date(created_time) >= '2024-01-01'
--                             AND     app IN ('xyf01') --
--                             AND     status = 2 --成功
--                             AND     inner_app = 'xyf01'
--                          and biz_flow_number  in (
--  -- 虚假给额的授信成功用户口径，biz_flow_number关联授信表
-- SELECT  biz_flow_number -- 授信biz_flow_number
--         -- ,id_card_number
--         -- ,GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") AS 人群类型
-- FROM    xyf_dwd.dwd_inloan_t_decision_result_detail_df
-- WHERE   pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
-- AND     enginecode = 'jcl_20240923000003'
-- -- AND     GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") = 'fake_activation'
-- and GET_JSON_OBJECT(context,"$.app_new_risk_mark_output") rlike 'fake_activation'
-- AND     decision_time >= '2024-10-15 00:00:00' --
-- -- 存在少量异常数据，同一个biz_flow_number+enginename 存在多条记录；下面排序做个兜底的清洗
-- QUALIFY ROW_NUMBER() OVER (PARTITION BY biz_flow_number,date(decision_time) ORDER BY decision_time DESC ) = 1
--                     )
--                     QUALIFY row_number() OVER (PARTITION BY cust_no order by created_time DESC ) = 1
--            ) g on shouxin.id_card_number =g.id_card_number and  shouxin.app =g.app 
--            --and rank_no=1 
--     left join 
--     (SELECT 
--         day_id_iso,
--         concat(day_week01_xf_new,'至',day_weekend_xf_new) as week_range
--     FROM
--         xyf_dim.dim_pub_date 
--     ) zhou on to_date(shouxin.row_crt_ts) = zhou.day_id_iso 
-- WHERE
-- 	date( shouxin.row_crt_ts )>= '2022-12-30' 
-- GROUP BY
-- 	date(shouxin.row_crt_ts) ,
-- 	zhou.week_range ,
--     substr( shouxin.row_crt_ts,1,7 ) ,
-- 	-- 授信额度  
--  case when    shouxin.shouxin_cnt=1 and 0<=g.final_amt and g.final_amt <=3000 then '1. 0k-3k'
-- 	      when shouxin.shouxin_cnt=1 and 3000<g.final_amt and g.final_amt <=5000 then '2. 3k-5k'
--                   when   shouxin.shouxin_cnt=1 and 5000<g.final_amt and g.final_amt<=10000 then '3. 5k-1w'
--                   when   shouxin.shouxin_cnt=1 and 10000<g.final_amt and g.final_amt<=20000 then '4. 1w-2w'
--                   when    shouxin.shouxin_cnt=1 and 20000<g.final_amt  and g.final_amt <=50000   then '5. 2w-5w'
--                   when    shouxin.shouxin_cnt=1 and 50000<g.final_amt   then '6. 5w+'
--                   else ''
--             end,
-- --  注册渠道
-- qu_zhu.first_channel ,
-- qu_gui.first_channel ,      
-- -- 完件评级
-- v1_score ,
-- ping2.score ,
-- -- v2.personalloan_sd_line_amtlevel_rh_app ,
-- v2.personalloan_sd_line_amtlevel_rh_app ,
--  case when v2.personalloan_sd_line_amtlevel_rh_app>0 and v2.personalloan_sd_line_amtlevel_rh_app<2 then 1
--       when v2.personalloan_sd_line_amtlevel_rh_app>=2 and v2.personalloan_sd_line_amtlevel_rh_app<3 then 2
--       when v2.personalloan_sd_line_amtlevel_rh_app>=3 and v2.personalloan_sd_line_amtlevel_rh_app<4 then 3
--       when v2.personalloan_sd_line_amtlevel_rh_app>=4 and v2.personalloan_sd_line_amtlevel_rh_app<5 then 4
--       when v2.personalloan_sd_line_amtlevel_rh_app>=5 and v2.personalloan_sd_line_amtlevel_rh_app<6 then 5
--       when v2.personalloan_sd_line_amtlevel_rh_app>=6 and v2.personalloan_sd_line_amtlevel_rh_app<7 then 6
--       when v2.personalloan_sd_line_amtlevel_rh_app>=7 and v2.personalloan_sd_line_amtlevel_rh_app<8 then 7
--       when v2.personalloan_sd_line_amtlevel_rh_app>=8  then '8+'
--       else ''
--  end ,
-- shouxin.shouxin_type ,
-- shouxin.yewu_type ,
-- case when  c.current_utm_source   like '%-DX-%' OR  c.current_utm_source   like '%DX-XYF01-TPKJ%'  OR  c.current_utm_source   like '%腾云天下%'   then n.name   else  '' end ,
-- case when datediff(shouxin.row_crt_ts,c.created_time) =0 then '0d'
-- 	when 0<datediff(shouxin.row_crt_ts,c.created_time) and datediff(shouxin.row_crt_ts,c.created_time)<=30 then '1-30d'
--     when datediff(shouxin.row_crt_ts,c.created_time)>30 and datediff(shouxin.row_crt_ts,c.created_time)<=60 then '31d-60d'
-- 	when datediff(shouxin.row_crt_ts,c.created_time)>60 then '61d+'
-- 	 else '' 
--      end 



